# Notebook 01 — Data Preparation

## Purpose

Establish a verified foundation for the NIRA digital policy feedback study before any topic modelling or supervised classification.

This notebook loads the source files, checks their structure, and records data-quality and linkage decisions. It does not modify the source files.

## Inputs

- `NIRA_Facebook_Datasets_5292.csv` — complete raw Facebook comment corpus.
- `Annotation Dataset/NIRA_Annotation_Master_1199_Modeling_final.xlsx` — manually annotated comments.
- `Publication Date Verification.xlsx` — working post register and government-response register.

## Outputs

A run folder under `outputs/01_data_preparation/` containing quality summaries, a linkage audit, a text-mismatch register, a publication-register audit, a working copy of the corpus, and a file manifest.

## Rules applied in this notebook

- `collection_date` is the time the comment file was collected. It is not the source post's publication date.
- `comment_id` is interpreted together with `post_url`. Repeated identifiers are not treated as duplicate comments unless the text and post also match.
- No raw records are deleted.
- Original comment text is preserved. Whitespace normalisation is stored as a separate field and is not a modelling decision.
- Rows that look as if they contain a phone number are flagged. Excerpts written by this notebook redact those numbers.


## 0. Set up the run

Locate the project, import the shared configuration, and choose the output folder for this run.


In [1]:
from pathlib import Path
import hashlib
import json
import re
import sys
from datetime import datetime, timezone

import pandas as pd

# Locate the project root from the working directory or this file's typical layout.
candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError(
        "Could not find NIRA_Facebook_Datasets_5292.csv. "
        "Run this notebook with the project folder as the working directory."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.project_config import (
    ANNOTATION_COLUMNS,
    ANNOTATION_XLSX,
    CLASSIFICATION_TARGETS,
    EXPECTED_ANNOTATION_N,
    EXPECTED_N_POSTS,
    EXPECTED_RAW_N,
    PUBLICATION_XLSX,
    RANDOM_SEED,
    RAW_COLUMNS,
    RAW_CORPUS_CSV,
)

RUN_ID = "20261018_content_review"
OUT = ROOT / "outputs" / "01_data_preparation" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Output directory:", OUT)
print("Random seed (reserved for later notebooks):", RANDOM_SEED)


Project root: D:\NIRA_CASE_STUDY
Output directory: D:\NIRA_CASE_STUDY\outputs\01_data_preparation\20261018_content_review
Random seed (reserved for later notebooks): 42


### Interpretation

The project root is `D:\NIRA_CASE_STUDY`. This run writes to `outputs/01_data_preparation/20261018_content_review`. Seed 42 is stored for the later notebooks. The source files are not changed.


## 1. Confirm that the source files exist

The notebook stops if a required source file is missing. File hashes are recorded so later runs can show whether a source changed. Nothing is written back to these paths.


In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


sources = {
    "raw_corpus": RAW_CORPUS_CSV,
    "annotation": ANNOTATION_XLSX,
    "publication_workbook": PUBLICATION_XLSX,
}
source_manifest = {}
for name, path in sources.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing source file: {path}")
    source_manifest[name] = {
        "path": str(path.relative_to(ROOT)),
        "bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    print(f"{name}: {path.name} ({path.stat().st_size:,} bytes)")


raw_corpus: NIRA_Facebook_Datasets_5292.csv (824,266 bytes)
annotation: NIRA_Annotation_Master_1199_Modeling_final.xlsx (138,746 bytes)
publication_workbook: Publication Date Verification.xlsx (17,207 bytes)


### Interpretation

The raw corpus, the annotation workbook, and the publication workbook are all present. The printed hashes identify the exact files read by this run. A later edit to a source file would change its hash.


## 2. Load and validate the raw corpus

The expected size is 5,292 comment records and 19 source posts. Columns are checked before any derived field is added.


In [3]:
raw = pd.read_csv(RAW_CORPUS_CSV, dtype=str, keep_default_na=False)
print("Raw rows:", len(raw))
print("Raw columns:", list(raw.columns))

missing_cols = [col for col in RAW_COLUMNS if col not in raw.columns]
if missing_cols:
    raise ValueError(f"Raw corpus is missing columns: {missing_cols}")
if len(raw) != EXPECTED_RAW_N:
    raise ValueError(f"Expected {EXPECTED_RAW_N} raw rows, found {len(raw)}")

raw = raw[RAW_COLUMNS].copy()
raw.insert(0, "source_row", range(1, len(raw) + 1))
raw["post_url_original"] = raw["post_url"]
raw["post_url_stripped"] = raw["post_url"].str.strip()
raw["record_key"] = raw["post_url_stripped"] + "||" + raw["comment_id"].str.strip()
raw["comment_original"] = raw["comment"]
raw["comment_whitespace_normalized"] = (
    raw["comment"].str.replace(r"\s+", " ", regex=True).str.strip()
)
raw["comment_char_len"] = raw["comment"].str.len()
raw["collection_date_parsed"] = pd.to_datetime(raw["collection_date"], errors="coerce")

if raw["collection_date_parsed"].isna().any():
    raise ValueError("One or more collection_date values could not be parsed.")
if raw["record_key"].duplicated().any():
    raise ValueError("record_key is not unique. Do not continue until the collision is documented.")

print("Unique record_key:", raw["record_key"].nunique())
print("Unique stripped post URLs:", raw["post_url_stripped"].nunique())
print("Collection timestamps that failed to parse: 0")


Raw rows: 5292
Raw columns: ['comment_id', 'comment', 'platform', 'post_url', 'collection_date']
Unique record_key: 5292
Unique stripped post URLs: 19
Collection timestamps that failed to parse: 0


### Interpretation

The raw file contains 5,292 comments and the five expected columns. `record_key`, the stripped post URL plus `comment_id`, is unique for every row. Nineteen distinct posts are present, and every collection timestamp parses. `comment_id` by itself is not the join key.


## 3. Raw-corpus quality checks

These checks describe the file. They do not delete rows.

`comment_id` restarts at 1 within each source post, so the same identifier appears on different posts. A comment is treated as a separate record when the post URL differs.


In [4]:
PHONE_LIKE = re.compile(
    r"(?i)(?:\+?252[\s\-]?\d[\d\s\-]{6,}\d|\btell\s*:\s*\+?\d[\d\s\-]+|\b\d{10,15}\b)"
)
ARABIC = re.compile(r"[\u0600-\u06FF]")
STYLED_LATIN = re.compile(r"[\U0001D400-\U0001D7FF]")


def redact_phone(text: str) -> str:
    return PHONE_LIKE.sub("[PHONE REDACTED]", text)


raw["phone_like_flag"] = raw["comment"].map(lambda text: bool(PHONE_LIKE.search(text)))
raw["arabic_script_flag"] = raw["comment"].map(lambda text: bool(ARABIC.search(text)))
raw["styled_latin_flag"] = raw["comment"].map(lambda text: bool(STYLED_LATIN.search(text)))
raw["has_newline"] = raw["comment"].str.contains(r"[\r\n]", regex=True)

within_post_id_dups = int(raw.duplicated(["post_url_stripped", "comment_id"]).sum())
within_post_text_dups = int(raw.duplicated(["post_url_stripped", "comment"]).sum())
cross_post_text_rows = int(raw.duplicated(["comment"], keep=False).sum())

urls_with_outer_space = int((raw["post_url"] != raw["post_url_stripped"]).sum())
blank_comments = int((raw["comment"].str.strip() == "").sum())
full_row_dups = int(raw.duplicated(RAW_COLUMNS).sum())

quality = {
    "raw_rows": int(len(raw)),
    "columns": RAW_COLUMNS,
    "blank_comments": blank_comments,
    "empty_cells_by_column": {
        col: int((raw[col].str.strip() == "").sum()) for col in RAW_COLUMNS
    },
    "platform_values": raw["platform"].value_counts().to_dict(),
    "full_row_duplicates": full_row_dups,
    "unique_comment_id": int(raw["comment_id"].nunique()),
    "duplicate_comment_id_rows": int(raw["comment_id"].duplicated().sum()),
    "within_post_duplicate_comment_id": within_post_id_dups,
    "within_post_duplicate_comment_text": within_post_text_dups,
    "rows_sharing_comment_text_with_another_post": cross_post_text_rows,
    "unique_comment_text": int(raw["comment"].nunique()),
    "unique_posts_after_url_strip": int(raw["post_url_stripped"].nunique()),
    "rows_with_outer_whitespace_in_post_url": urls_with_outer_space,
    "unique_collection_timestamps": int(raw["collection_date"].nunique()),
    "collection_date_min": raw["collection_date_parsed"].min().isoformat(sep=" "),
    "collection_date_max": raw["collection_date_parsed"].max().isoformat(sep=" "),
    "comment_char_len": {
        "min": int(raw["comment_char_len"].min()),
        "p25": float(raw["comment_char_len"].quantile(0.25)),
        "median": float(raw["comment_char_len"].median()),
        "p75": float(raw["comment_char_len"].quantile(0.75)),
        "max": int(raw["comment_char_len"].max()),
        "mean": float(raw["comment_char_len"].mean()),
    },
    "comments_under_30_characters": int((raw["comment_char_len"] < 30).sum()),
    "comments_over_280_characters": int((raw["comment_char_len"] > 280).sum()),
    "comments_with_newline": int(raw["has_newline"].sum()),
    "comments_with_arabic_script": int(raw["arabic_script_flag"].sum()),
    "comments_with_styled_latin_letters": int(raw["styled_latin_flag"].sum()),
    "comments_with_phone_like_pattern": int(raw["phone_like_flag"].sum()),
    "decision_comment_id": (
        "comment_id is unique within each stripped post_url and restarts at 1 on every post. "
        "Global repetition is not sufficient evidence of a duplicate comment. No rows were removed."
    ),
    "decision_post_url_whitespace": (
        "212 rows on one URL have a leading space. The original URL is retained. "
        "Joins use post_url_stripped."
    ),
    "decision_collection_date": (
        "Each of the 19 posts has a single collection timestamp. "
        "These values describe data collection, not source-post publication."
    ),
    "decision_text_normalization": (
        "comment_original keeps the file text. comment_whitespace_normalized collapses "
        "whitespace only. Case, Somali spelling, and tokens are unchanged at this stage."
    ),
}

if quality["unique_posts_after_url_strip"] != EXPECTED_N_POSTS:
    raise ValueError("Post count after URL stripping does not equal 19.")
if within_post_id_dups != 0 or within_post_text_dups != 0 or full_row_dups != 0 or blank_comments != 0:
    raise ValueError("An unexpected within-post duplicate or blank comment was found.")

print("Blank comments:", blank_comments)
print("Full-row duplicates:", full_row_dups)
print("comment_id unique globally:", quality["unique_comment_id"], "of", len(raw))
print("Within-post duplicate comment_id:", within_post_id_dups)
print("Rows whose text also appears on another post:", cross_post_text_rows)
print("URL rows with outer whitespace:", urls_with_outer_space)
print("Phone-like comments (flagged, not printed):", quality["comments_with_phone_like_pattern"])
print("Arabic-script comments:", quality["comments_with_arabic_script"])
print("Styled-Latin comments:", quality["comments_with_styled_latin_letters"])


Blank comments: 0
Full-row duplicates: 0
comment_id unique globally: 1013 of 5292
Within-post duplicate comment_id: 0
Rows whose text also appears on another post: 15
URL rows with outer whitespace: 212
Phone-like comments (flagged, not printed): 11
Arabic-script comments: 7
Styled-Latin comments: 5


### Interpretation

No comment is blank and no full row is duplicated. `comment_id` has only 1,013 distinct values because it restarts inside each post. Fifteen rows repeat text that also appears under another post, and those rows are kept. Eleven comments match the phone pattern and are flagged rather than printed. Seven comments use Arabic script and five use styled Latin letters.


## 4. Source posts and collection dates

Comment counts below are counts of comment records. They are not counts of source posts. The date is the collection timestamp stored with that post's comments.


In [5]:
post_summary = (
    raw.groupby("post_url_stripped", as_index=False)
    .agg(
        n_comments=("comment_id", "size"),
        unique_comment_id=("comment_id", "nunique"),
        collection_date=("collection_date", "first"),
        collection_date_parsed=("collection_date_parsed", "first"),
        n_collection_timestamps=("collection_date", "nunique"),
        rows_with_raw_url_whitespace=("post_url", lambda s: int((s != s.str.strip()).sum())),
        median_comment_chars=("comment_char_len", "median"),
        phone_like_comments=("phone_like_flag", "sum"),
    )
    .sort_values("n_comments", ascending=False)
)
post_summary["share_of_corpus"] = post_summary["n_comments"] / len(raw)
print(post_summary[["post_url_stripped", "n_comments", "collection_date"]].to_string(index=False))
print("Posts:", len(post_summary))
print("Collection dates are constant within each post:", bool((post_summary["n_collection_timestamps"] == 1).all()))


                                                         post_url_stripped  n_comments collection_date
                            https://www.facebook.com/reel/1987644858636653        1013  8/8/2026 22:45
                              https://www.facebook.com/share/v/195WLJuBek/         660 7/24/2026 20:27
                              https://www.facebook.com/share/v/1Dz3cCVoam/         642  8/8/2026 23:21
                              https://www.facebook.com/share/r/1BpLntUmD3/         505  7/23/2026 7:50
                              https://www.facebook.com/share/v/1KyVGPC6xD/         320 7/24/2026 20:54
                             https://www.facebook.com/share/p/14jzMqGJjmG/         317 7/24/2026 21:30
                              https://www.facebook.com/share/v/199hF8Strp/         267 7/24/2026 19:32
                              https://www.facebook.com/share/p/1cn5QHogbW/         236 7/24/2026 21:41
                              https://www.facebook.com/share/p/1EpQdDQavp

### Interpretation

Each of the 19 posts has one collection timestamp. That timestamp is the time the comment file was collected. It is not the publication date of the Facebook post. The largest post contains 1,013 comments.


## 5. Manually annotated records

The annotation file is checked for the expected 1,199 rows, the four classification fields, and any missing labels. `actor` is present in the file but is not one of the four supervised tasks defined for this project.


In [6]:
annotation = pd.read_excel(ANNOTATION_XLSX, dtype=str)
print("Annotation rows:", len(annotation))
print("Annotation columns:", list(annotation.columns))

missing_ann_cols = [col for col in ANNOTATION_COLUMNS if col not in annotation.columns]
if missing_ann_cols:
    raise ValueError(f"Annotation file is missing columns: {missing_ann_cols}")
if len(annotation) != EXPECTED_ANNOTATION_N:
    raise ValueError(f"Expected {EXPECTED_ANNOTATION_N} annotated rows, found {len(annotation)}")

annotation = annotation[ANNOTATION_COLUMNS].copy()
for col in ANNOTATION_COLUMNS:
    annotation[col] = annotation[col].fillna("").astype(str)

empty_labels = {
    col: int((annotation[col].str.strip() == "").sum())
    for col in CLASSIFICATION_TARGETS + ["actor", "annotation_status", "annotation_id", "comment"]
}
if any(empty_labels.values()):
    raise ValueError(f"Unexpected empty annotation fields: {empty_labels}")
if annotation["annotation_id"].duplicated().any():
    raise ValueError("annotation_id is not unique.")

annotation["post_url_stripped"] = annotation["post_url"].str.strip()
annotation["record_key"] = annotation["post_url_stripped"] + "||" + annotation["comment_id"].str.strip()
if annotation["record_key"].duplicated().any():
    raise ValueError("Annotation record_key is not unique.")

label_frames = []
for col in CLASSIFICATION_TARGETS + ["actor", "annotation_status"]:
    counts = annotation[col].value_counts(dropna=False).rename_axis("label").reset_index(name="n_comments")
    counts.insert(0, "field", col)
    counts["share_of_annotated"] = counts["n_comments"] / len(annotation)
    label_frames.append(counts)
label_counts = pd.concat(label_frames, ignore_index=True)
print(label_counts.to_string(index=False))
print("Unique annotation post URLs after stripping:", annotation["post_url_stripped"].nunique())


Annotation rows: 1199
Annotation columns: ['annotation_id', 'comment_id', 'comment', 'platform', 'post_url', 'collection_date', 'collection_date_original', 'policy_relevance', 'feedback_type', 'topic', 'sentiment', 'actor', 'annotation_status']
            field                         label  n_comments  share_of_annotated
 policy_relevance                      relevant         821            0.684737
 policy_relevance                  not_relevant         237            0.197665
 policy_relevance                     uncertain         141            0.117598
    feedback_type                     complaint         317            0.264387
    feedback_type                         other         232            0.193495
    feedback_type                       support         172            0.143453
    feedback_type                      question         162            0.135113
    feedback_type                       request         153            0.127606
    feedback_type                  

### Interpretation

The annotation workbook contains 1,199 rows. Every classification field is filled, and every row is marked annotated. Inside that subset the largest classes are `relevant`, `complaint`, topic `other`, and `negative`. These counts do not describe the other 4,093 comments.


## 6. Relationship between the annotated records and the raw corpus

Matching uses the stripped post URL plus `comment_id`, because `comment_id` alone is not unique. Text is then compared. Differences are recorded. They are not silently corrected.


In [7]:
def whitespace_norm(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def classify_text(annotated_text: str, raw_text: str) -> tuple[str, str]:
    if annotated_text == raw_text:
        return "exact", "texts are identical"
    if whitespace_norm(annotated_text) == whitespace_norm(raw_text):
        return "newline_or_whitespace", "difference is limited to line breaks or spacing"
    ann_norm = whitespace_norm(annotated_text)
    raw_norm = whitespace_norm(raw_text)
    if "_x000D_" in annotated_text:
        return "content_difference", "annotation contains the literal Excel marker _x000D_"
    if "REDACTED" in ann_norm:
        return "content_difference", "annotation redacts material that is present in the raw comment"
    if raw_norm in ann_norm or ann_norm in raw_norm:
        return "content_difference", "one version contains the other plus extra characters"
    ann_core = ann_norm.rstrip(".,!")
    raw_core = raw_norm.rstrip(".,!")
    if ann_core == raw_core:
        return "content_difference", "trailing punctuation differs"
    if abs(len(ann_norm) - len(raw_norm)) <= 2:
        return "content_difference", "small character-level difference"
    return "content_difference", "texts differ after whitespace normalisation"


raw_for_join = raw[
    ["record_key", "source_row", "comment_original", "collection_date", "collection_date_parsed"]
].rename(
    columns={
        "comment_original": "raw_comment",
        "collection_date": "raw_collection_date",
        "collection_date_parsed": "raw_collection_date_parsed",
    }
)
linked = annotation.merge(raw_for_join, on="record_key", how="left", indicator=True)
match_counts = linked["_merge"].value_counts().to_dict()
if match_counts.get("both", 0) != len(annotation) or match_counts.get("left_only", 0) != 0:
    raise ValueError(f"Annotation keys did not all match the raw corpus: {match_counts}")

statuses = linked.apply(lambda row: classify_text(row["comment"], row["raw_comment"]), axis=1)
linked["text_match_status"] = statuses.map(lambda item: item[0])
linked["text_difference_note"] = statuses.map(lambda item: item[1])

# Collection dates: compare to the minute. String formats differ for one post.
ann_collection = pd.to_datetime(linked["collection_date"], errors="coerce")
collection_minute_mismatch = int(
    (ann_collection.dt.floor("min") != linked["raw_collection_date_parsed"].dt.floor("min")).sum()
)
original_vs_collection_string_mismatch = int(
    (annotation["collection_date"] != annotation["collection_date_original"]).sum()
)

# Coverage by post. Both columns are comment counts.
coverage = post_summary[["post_url_stripped", "n_comments"]].merge(
    annotation.groupby("post_url_stripped").size().rename("n_annotated").reset_index(),
    on="post_url_stripped",
    how="left",
)
coverage["n_annotated"] = coverage["n_annotated"].fillna(0).astype(int)
coverage["annotated_share_of_post"] = coverage["n_annotated"] / coverage["n_comments"]

# Selection gaps. This describes spacing; it does not prove a sampling design.
raw["is_annotated"] = raw["record_key"].isin(set(annotation["record_key"]))
raw["rank_in_post"] = raw.groupby("post_url_stripped").cumcount()
gaps = []
for _, group in raw.groupby("post_url_stripped"):
    ranks = group.loc[group["is_annotated"], "rank_in_post"].tolist()
    gaps.extend(ranks[i + 1] - ranks[i] for i in range(len(ranks) - 1))
gap_series = pd.Series(gaps)

linkage = {
    "annotated_rows": int(len(annotation)),
    "raw_rows": int(len(raw)),
    "annotated_share_of_raw_comments": len(annotation) / len(raw),
    "keys_matched": int(match_counts.get("both", 0)),
    "keys_unmatched": int(match_counts.get("left_only", 0)),
    "text_match_counts": linked["text_match_status"].value_counts().to_dict(),
    "collection_date_minute_mismatches_on_matched_keys": collection_minute_mismatch,
    "collection_date_string_differs_from_collection_date_original": original_vs_collection_string_mismatch,
    "collection_date_original_note": (
        "90 rows, all from one post, store collection_date_original as '7/23/2026 7:50' "
        "while collection_date is an ISO timestamp of the same minute. "
        "After parsing, every matched key agrees with the raw collection timestamp to the minute."
    ),
    "annotated_share_by_post_min": float(coverage["annotated_share_of_post"].min()),
    "annotated_share_by_post_max": float(coverage["annotated_share_of_post"].max()),
    "within_post_gap_between_annotated_rows": {
        "n_gaps": int(gap_series.shape[0]),
        "median": float(gap_series.median()),
        "mean": float(gap_series.mean()),
        "min": int(gap_series.min()),
        "max": int(gap_series.max()),
    },
    "sampling_note": (
        "About 22.7% of comments are annotated, and every post is represented at roughly 22–23%. "
        "The gap between annotated rows is not a single fixed stride. "
        "No sampling protocol is stored in the project files, so the design cannot be confirmed."
    ),
    "overlap_note": (
        "The 1,199 annotated records are a matched subset of the 5,292 raw comments. "
        "They are not a separate population."
    ),
}

print("Key matches:", linkage["keys_matched"], "unmatched:", linkage["keys_unmatched"])
print("Text match counts:", linkage["text_match_counts"])
print("Collection-minute mismatches:", collection_minute_mismatch)
print(
    "Annotated share of each post ranges from",
    round(linkage["annotated_share_by_post_min"], 3),
    "to",
    round(linkage["annotated_share_by_post_max"], 3),
)
print(linkage["sampling_note"])


Key matches: 1199 unmatched: 0
Text match counts: {'exact': 1174, 'content_difference': 13, 'newline_or_whitespace': 12}
Collection-minute mismatches: 0
Annotated share of each post ranges from 0.219 to 0.233
About 22.7% of comments are annotated, and every post is represented at roughly 22–23%. The gap between annotated rows is not a single fixed stride. No sampling protocol is stored in the project files, so the design cannot be confirmed.


### Interpretation

All 1,199 annotation keys match a raw comment. The text is an exact match on 1,174 rows. Twelve rows differ only by spacing or line breaks, and 13 differ in characters. Those 25 differences are retained. Each post contributes about 22 to 23 percent of its comments. No sampling protocol is stored in the files.


## 7. Publication-date workbook

The Post Register includes `post_url` and the verification-status fields. A separate sheet, `Content Evidence Review`, records the current working description for POST_17 and POST_18.

This notebook joins `post_url` to the raw corpus. A match attaches the register date and the status fields to each comment. `collection_date` remains the collection timestamp. The workbook records publication dates as researcher-verified. A confirmed URL or date does not establish a government response.

POST_17 is a likely topic-level description of NIRA card issuance through the Somali Embassy in Uganda. POST_18 is a probable Eid greeting inferred from comments and the date. Neither original Facebook caption was directly recovered.


In [8]:
publication_book = pd.ExcelFile(PUBLICATION_XLSX)
print("Sheets:", publication_book.sheet_names)

post_register = pd.read_excel(PUBLICATION_XLSX, sheet_name="Post Register", dtype=str)
response_register = pd.read_excel(PUBLICATION_XLSX, sheet_name="Response Register", dtype=str)
readme = pd.read_excel(PUBLICATION_XLSX, sheet_name="Read Me and Limitations", dtype=str)
if "Content Evidence Review" not in publication_book.sheet_names:
    raise ValueError("Publication workbook is missing the Content Evidence Review sheet.")
content_review = pd.read_excel(PUBLICATION_XLSX, sheet_name="Content Evidence Review", dtype=str)
print("Content evidence review rows:", len(content_review), "columns:", list(content_review.columns))

required_register_columns = [
    "post_id",
    "account_name",
    "publication_date",
    "post_url",
    "evidence_status",
    "publication_date_status",
    "post_url_status",
    "content_evidence_status",
    "verification_basis",
]
missing_register_columns = [col for col in required_register_columns if col not in post_register.columns]
if missing_register_columns:
    raise ValueError(f"Post Register is missing columns: {missing_register_columns}")

print("Post register rows:", len(post_register), "columns:", list(post_register.columns))
print("Response register rows:", len(response_register), "columns:", list(response_register.columns))

post_register["post_url_stripped"] = post_register["post_url"].fillna("").astype(str).str.strip()
empty_urls = int((post_register["post_url_stripped"] == "").sum())
duplicate_urls = int(post_register["post_url_stripped"].duplicated().sum())

comment_counts = raw.groupby("post_url_stripped").size().rename("n_comments")
url_crosswalk = post_register.merge(
    comment_counts,
    left_on="post_url_stripped",
    right_index=True,
    how="outer",
    indicator=True,
)
url_crosswalk["n_comments"] = url_crosswalk["n_comments"].fillna(0).astype(int)
matched = int((url_crosswalk["_merge"] == "both").sum())
register_only = int((url_crosswalk["_merge"] == "left_only").sum())
corpus_only = int((url_crosswalk["_merge"] == "right_only").sum())
comment_sum = int(url_crosswalk.loc[url_crosswalk["_merge"] == "both", "n_comments"].sum())

if empty_urls or duplicate_urls or register_only or corpus_only or matched != EXPECTED_N_POSTS or comment_sum != len(raw):
    raise ValueError(
        "URL join failed: "
        f"empty={empty_urls}, duplicate={duplicate_urls}, matched={matched}, "
        f"register_only={register_only}, corpus_only={corpus_only}, comments={comment_sum}"
    )

previous_path = ROOT / "outputs" / "01_data_preparation" / "20261018_verified_register" / "post_url_crosswalk.csv"
changes_since_previous = []
if previous_path.exists():
    previous = pd.read_csv(previous_path, dtype=str)
    compared = previous.merge(post_register, on="post_id", suffixes=("_previous", "_current"))
    for field in [
        "account_name",
        "publication_date",
        "post_url",
        "policy_topic",
        "evidence_status",
        "linkage_note",
        "content_evidence_status",
        "verification_basis",
    ]:
        previous_field = f"{field}_previous"
        current_field = f"{field}_current"
        changed = compared.loc[
            compared[previous_field].fillna("") != compared[current_field].fillna(""),
            ["post_id", previous_field, current_field],
        ]
        for _, row in changed.iterrows():
            changes_since_previous.append(
                {
                    "post_id": row["post_id"],
                    "field": field,
                    "previous": row[previous_field],
                    "current": row[current_field],
                }
            )

content_review_fields = [
    "post_id",
    "resolution_status",
    "working_content_description",
    "evidence_type",
    "confidence",
    "guidance_for_analysis",
    "remaining_limitation",
    "source_url_1",
    "source_url_2",
]
missing_review_fields = [col for col in content_review_fields if col not in content_review.columns]
if missing_review_fields:
    raise ValueError(f"Content Evidence Review is missing columns: {missing_review_fields}")
url_crosswalk = url_crosswalk.merge(
    content_review[content_review_fields],
    on="post_id",
    how="left",
)

phrase_checks = {
    "POST_17": {
        "url": "https://www.facebook.com/share/v/1DGzDhdKuS/",
        "phrases": ["uganda", "safaarad", "embassy"],
    },
    "POST_18": {
        "url": "https://www.facebook.com/photo/?fbid=933559576313414&set=a.171201679215878",
        "phrases": [
            "Ciidiina marabnee kaararka meel ku sheega",
            "Card ka nasiiya ciidina cune",
            "ciid",
        ],
    },
}
comment_corroboration = {}
for post_id, spec in phrase_checks.items():
    comments = raw.loc[raw["post_url_stripped"] == spec["url"], "comment"]
    comment_corroboration[post_id] = {
        "n_comments": int(len(comments)),
        "phrase_hits": {
            phrase: int(comments.str.contains(phrase, case=False, regex=False).sum())
            for phrase in spec["phrases"]
        },
    }
cited_files_absent = [
    "NIRA_Manual_Annotation_Dataset_Annotated_REVISED.csv",
    "NIRA_AI_PreAnnotated_Sample_300.csv",
]
cited_files_absent = [name for name in cited_files_absent if not (ROOT / name).exists() and not list(ROOT.rglob(name))]

publication_audit = {
    "sheets": publication_book.sheet_names,
    "post_register_rows": int(len(post_register)),
    "post_register_columns": [col for col in post_register.columns if col != "post_url_stripped"],
    "response_register_rows": int(len(response_register)),
    "response_register_columns": list(response_register.columns),
    "post_register_contains_url_column": True,
    "urls_matched_to_raw_corpus": matched,
    "urls_in_register_only": register_only,
    "urls_in_corpus_only": corpus_only,
    "comments_covered_by_matched_urls": comment_sum,
    "readme_notes": readme.fillna("").astype(str).to_dict(orient="records"),
    "publication_date_status_counts": post_register["publication_date_status"].value_counts().to_dict(),
    "post_url_status_counts": post_register["post_url_status"].value_counts().to_dict(),
    "changes_since_20261018_verified_register": changes_since_previous,
    "content_evidence_review": content_review.fillna("").to_dict(orient="records"),
    "comment_corroboration_in_raw_corpus": comment_corroboration,
    "cited_evidence_files_not_in_project": cited_files_absent,
    "join_to_comment_corpus": (
        "The Post Register column post_url matches all 19 stripped URLs in the raw corpus. "
        "Every comment can be attached to exactly one post_id."
    ),
    "authority": (
        "The workbook records all 19 publication dates and post URLs as researcher-verified. "
        "POST_08 is dated 2026-07-06. "
        "POST_17 is recorded as likely identified at topic level: NIRA national-ID issuance through the Somali Embassy in Uganda, "
        "with an SNTV report dated 2025-12-25 and comments on the matched URL. "
        "The original Facebook media item was not directly retrieved. "
        "POST_18 is recorded as a probable Eid al-Fitr greeting inferred from comments and the 2026-03-19 date, "
        "the day before Eid on 2026-03-20. The original image and caption were not recovered. "
        "A confirmed URL or date does not establish a direct government response."
    ),
}

print("Matched URLs:", matched, "register-only:", register_only, "corpus-only:", corpus_only)
print("Comments covered:", comment_sum)
print("Changes since 20261018_verified_register:", len(changes_since_previous))
print("Comment corroboration:", comment_corroboration)
print("Cited files absent:", cited_files_absent)
print(
    url_crosswalk.loc[url_crosswalk["_merge"] == "both"]
    .sort_values("publication_date")[
        ["post_id", "publication_date", "account_name", "n_comments", "publication_date_status"]
    ]
    .to_string(index=False)
)


Sheets: ['Post Register', 'Response Register', 'Read Me and Limitations', 'Content Evidence Review']
Content evidence review rows: 2 columns: ['post_id', 'publication_date', 'account_name', 'post_url', 'resolution_status', 'working_content_description', 'evidence_summary', 'evidence_type', 'source_url_1', 'source_url_2', 'confidence', 'guidance_for_analysis', 'remaining_limitation']
Post register rows: 19 columns: ['post_id', 'account_name', 'publication_date', 'summary', 'policy_topic', 'public_feedback_issue', 'government_response', 'evidence_status', 'linkage_note', 'post_url', 'publication_date_status', 'post_url_status', 'content_evidence_status', 'verification_basis']
Response register rows: 5 columns: ['response_id', 'source', 'date', 'title', 'topic', 'url', 'evidence_status']
Matched URLs: 19 register-only: 0 corpus-only: 0
Comments covered: 5292
Changes since 20261018_verified_register: 10
Comment corroboration: {'POST_17': {'n_comments': 73, 'phrase_hits': {'uganda': 3, 'saf

### Interpretation

All 19 register URLs match the comment file and together account for all 5,292 comments. The publication dates are the researcher-verified register values, including POST_08 on 2026-07-06. A matched URL or date does not establish that an institution responded to a comment.


## 8. Save the working files and verify them

The working corpus adds keys and flags. It keeps every raw row and the original comment. Phone-like strings are redacted only in the mismatch excerpts, not in the working corpus, because later text models need the source wording. The phone-like flag marks those rows for careful handling in qualitative extracts.


In [9]:
mismatch = linked.loc[linked["text_match_status"] != "exact", [
    "annotation_id",
    "record_key",
    "source_row",
    "text_match_status",
    "text_difference_note",
    "comment",
    "raw_comment",
]].copy()
mismatch["comment_redacted"] = mismatch["comment"].map(redact_phone)
mismatch["raw_comment_redacted"] = mismatch["raw_comment"].map(redact_phone)
mismatch = mismatch.drop(columns=["comment", "raw_comment"])

cross_post = raw.loc[raw.duplicated("comment", keep=False), [
    "source_row",
    "comment_id",
    "post_url_stripped",
    "collection_date",
    "comment",
]].sort_values(["comment", "post_url_stripped"])
cross_post["comment"] = cross_post["comment"].map(redact_phone)

working_columns = [
    "source_row",
    "record_key",
    "comment_id",
    "comment_original",
    "comment_whitespace_normalized",
    "platform",
    "post_url_original",
    "post_url_stripped",
    "collection_date",
    "collection_date_parsed",
    "comment_char_len",
    "has_newline",
    "phone_like_flag",
    "arabic_script_flag",
    "styled_latin_flag",
    "is_annotated",
]
working = raw[working_columns].copy()
working["collection_date_parsed"] = working["collection_date_parsed"].dt.strftime("%Y-%m-%d %H:%M:%S")
register_fields = url_crosswalk.loc[
    url_crosswalk["_merge"] == "both",
    [
        "post_url_stripped",
        "post_id",
        "publication_date",
        "evidence_status",
        "account_name",
        "publication_date_status",
        "post_url_status",
        "content_evidence_status",
    ],
].rename(
    columns={
        "publication_date": "publication_date_register",
        "evidence_status": "publication_evidence_status",
        "account_name": "account_name_register",
    }
)
working = working.merge(register_fields, on="post_url_stripped", how="left")
if working["post_id"].isna().any() or len(working) != len(raw):
    raise ValueError("Attaching post_id to the working corpus failed.")
post_summary = post_summary.merge(register_fields, on="post_url_stripped", how="left")

linked_out = linked[[
    "annotation_id",
    "record_key",
    "source_row",
    "comment_id",
    "post_url_stripped",
    "policy_relevance",
    "feedback_type",
    "topic",
    "sentiment",
    "actor",
    "annotation_status",
    "text_match_status",
    "text_difference_note",
    "collection_date",
    "collection_date_original",
    "raw_collection_date",
]].copy()

written = {}

def save_df(frame: pd.DataFrame, name: str) -> None:
    path = OUT / name
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    written[name] = {"rows": int(len(frame)), "bytes": path.stat().st_size}

def save_json(payload: dict, name: str) -> None:
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    written[name] = {"bytes": path.stat().st_size}

save_df(working, "raw_corpus_working.csv")
save_df(post_summary.assign(
    collection_date_parsed=post_summary["collection_date_parsed"].dt.strftime("%Y-%m-%d %H:%M:%S")
), "post_level_summary.csv")
save_df(label_counts, "annotation_label_counts.csv")
save_df(coverage, "annotation_coverage_by_post.csv")
save_df(linked_out, "annotation_linked.csv")
save_df(mismatch, "text_mismatch_register.csv")
save_df(cross_post, "cross_post_repeated_text.csv")
save_df(post_register, "post_register_as_read.csv")
crosswalk_out = url_crosswalk.loc[
    url_crosswalk["_merge"] == "both",
    [
        "post_id",
        "account_name",
        "publication_date",
        "publication_date_status",
        "post_url_status",
        "content_evidence_status",
        "verification_basis",
        "evidence_status",
        "post_url",
        "post_url_stripped",
        "n_comments",
        "policy_topic",
        "linkage_note",
        "resolution_status",
        "working_content_description",
        "confidence",
        "remaining_limitation",
    ],
].sort_values("post_id")
save_df(crosswalk_out, "post_url_crosswalk.csv")
save_df(response_register, "response_register_as_read.csv")
save_df(content_review, "content_evidence_review.csv")
save_json(quality, "raw_quality_summary.json")
save_json(linkage, "linkage_summary.json")
save_json(publication_audit, "publication_register_audit.json")
save_json(source_manifest, "source_file_hashes.json")

report_lines = [
    "# Data-quality report — Notebook 01",
    "",
    f"Run id: `{RUN_ID}`",
    "",
    "This report was produced by `notebooks/01_Data_Preparation.ipynb`. Source files were not modified.",
    "",
    "## Raw corpus",
    "",
    f"- File: `{source_manifest['raw_corpus']['path']}`",
    f"- Rows: {quality['raw_rows']}",
    f"- Columns: {', '.join(quality['columns'])}",
    f"- Blank comments: {quality['blank_comments']}",
    f"- Exact duplicate rows: {quality['full_row_duplicates']}",
    f"- Source posts after stripping URL whitespace: {quality['unique_posts_after_url_strip']}",
    f"- `comment_id` values are unique within a post, but only {quality['unique_comment_id']} distinct identifiers exist across {quality['raw_rows']} rows.",
    f"- Rows whose exact text also appears under another post: {quality['rows_sharing_comment_text_with_another_post']}. These were retained.",
    f"- Rows with outer whitespace in `post_url`: {quality['rows_with_outer_whitespace_in_post_url']}.",
    f"- Collection timestamps: {quality['unique_collection_timestamps']}, from {quality['collection_date_min']} to {quality['collection_date_max']}.",
    "- Collection dates are not publication dates.",
    f"- Comment length in characters: minimum {quality['comment_char_len']['min']}, median {quality['comment_char_len']['median']}, maximum {quality['comment_char_len']['max']}.",
    f"- Comments under 30 characters: {quality['comments_under_30_characters']}.",
    f"- Comments containing Arabic script: {quality['comments_with_arabic_script']}.",
    f"- Comments containing styled Latin letters: {quality['comments_with_styled_latin_letters']}.",
    f"- Comments matching a phone-like pattern: {quality['comments_with_phone_like_pattern']}. Values are not printed in this report.",
    "",
    "## Annotated subset",
    "",
    f"- File: `{source_manifest['annotation']['path']}`",
    f"- Rows: {linkage['annotated_rows']}. All four classification fields are populated.",
    f"- Keys matched to the raw corpus on stripped URL plus `comment_id`: {linkage['keys_matched']}.",
    f"- Unmatched keys: {linkage['keys_unmatched']}.",
    f"- Text comparison: {linkage['text_match_counts']}.",
    f"- Annotated share of raw comments: {linkage['annotated_share_of_raw_comments']:.4f}.",
    f"- Annotated share within posts ranges from {linkage['annotated_share_by_post_min']:.3f} to {linkage['annotated_share_by_post_max']:.3f}.",
    f"- {linkage['sampling_note']}",
    f"- {linkage['overlap_note']}",
    "",
    "## Publication workbook",
    "",
    f"- Sheets: {', '.join(publication_audit['sheets'])}.",
    f"- Post register rows: {publication_audit['post_register_rows']}. Response register rows: {publication_audit['response_register_rows']}.",
    f"- URL matches: {publication_audit['urls_matched_to_raw_corpus']} of {publication_audit['post_register_rows']} register posts, covering {publication_audit['comments_covered_by_matched_urls']} comments.",
    f"- {publication_audit['join_to_comment_corpus']}",
    f"- {publication_audit['authority']}",
    f"- Changes since run 20261018_verified_register: {len(publication_audit['changes_since_20261018_verified_register'])} field changes, limited to POST_17 and POST_18 content fields.",
    f"- Comment corroboration in the raw corpus: {publication_audit['comment_corroboration_in_raw_corpus']}",
    f"- Evidence filenames cited by the workbook but not present in the project folder: {publication_audit['cited_evidence_files_not_in_project']}",
    "",
    "## Preprocessing decision",
    "",
    quality["decision_text_normalization"],
    "",
    "No modelling text was created beyond whitespace normalisation. Spelling normalisation, stopword removal, and translation were not applied.",
]
report_path = OUT / "DATA_QUALITY_REPORT.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
written["DATA_QUALITY_REPORT.md"] = {"bytes": report_path.stat().st_size}

manifest = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "notebook": "notebooks/01_Data_Preparation.ipynb",
    "sources": source_manifest,
    "outputs": written,
    "rows_removed": 0,
}
manifest_path = OUT / "MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Wrote", len(written), "output files to", OUT)
for name, info in written.items():
    path = OUT / name
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Output missing or empty: {path}")
    print(f"  OK {name} ({path.stat().st_size:,} bytes)")

# Guard: source hashes unchanged.
for name, path in sources.items():
    if sha256(path) != source_manifest[name]["sha256"]:
        raise RuntimeError(f"Source file changed during the run: {path}")
print("Source hashes unchanged.")
print("Working corpus rows:", len(pd.read_csv(OUT / "raw_corpus_working.csv")))
print("Linked annotation rows:", len(pd.read_csv(OUT / "annotation_linked.csv")))


Wrote 16 output files to D:\NIRA_CASE_STUDY\outputs\01_data_preparation\20261018_content_review
  OK raw_corpus_working.csv (4,278,312 bytes)
  OK post_level_summary.csv (10,412 bytes)
  OK annotation_label_counts.csv (1,379 bytes)
  OK annotation_coverage_by_post.csv (1,435 bytes)
  OK annotation_linked.csv (310,142 bytes)
  OK text_mismatch_register.csv (12,710 bytes)
  OK cross_post_repeated_text.csv (1,739 bytes)
  OK post_register_as_read.csv (22,501 bytes)
  OK post_url_crosswalk.csv (17,039 bytes)
  OK response_register_as_read.csv (1,632 bytes)
  OK content_evidence_review.csv (2,510 bytes)
  OK raw_quality_summary.json (1,939 bytes)
  OK linkage_summary.json (1,322 bytes)
  OK publication_register_audit.json (13,514 bytes)
  OK source_file_hashes.json (582 bytes)
  OK DATA_QUALITY_REPORT.md (3,803 bytes)
Source hashes unchanged.
Working corpus rows: 5292
Linked annotation rows: 1199


### Interpretation

The working corpus, the linked annotation table, the post crosswalk, and the quality report were written and then read back. Later notebooks start from this folder. The three original source files remain unchanged.


## 9. What this notebook does not claim

- It does not estimate themes, sentiment, or policy relevance for the unannotated comments.
- It does not treat the annotated labels as labels for the other 4,093 comments.
- It attaches each comment to the register publication date through the matched Facebook URL. The workbook labels those dates researcher-verified. This notebook does not add a further external check.
- `collection_date` is still the collection timestamp. It is stored separately from `publication_date_register`.
- POST_17 may be used as a qualified topic-level description of embassy-based NIRA card issuance. The original Facebook caption or video was not directly recovered.
- POST_18 may be used as a probable Eid greeting inferred from comments and the date. The original image and caption were not recovered, so the post text cannot be quoted.
- Confirming a source URL or publication date does not establish a direct government response.
- The 25 text differences remain unresolved differences between the annotation file and the raw file. Later notebooks should use `record_key` and keep both texts available.

The next notebook, `02_EDA.ipynb`, should use `raw_corpus_working.csv`, `annotation_linked.csv`, `post_url_crosswalk.csv`, and `content_evidence_review.csv` from this run.
